# Снижение активности по картам: кто, насколько и за счёт чего

Автор: Анна Суходолова  
Период: январь 2024 — июнь 2025  
Для департамента розничных продуктов банка «Меридиан»

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style='whitegrid', font='DejaVu Sans', font_scale=.85)
plt.rcParams.update({'figure.dpi':110, 'axes.spines.top':False, 'axes.spines.right':False})

def fmt(value, digits=1):
    return '—' if pd.isna(value) else f'{value:,.{digits}f}'.replace(',', ' ').replace('.', ',')

def show_table(frame, digits=2):
    display(frame.round(digits))

def flag(series):
    return series.astype('string').str.strip().str.casefold().isin(['true','1','да'])

# Читаем готовые данные. Сырьё не очищаем, назначения не меняем.
folder=Path('data/processed')
clean=pd.read_csv(folder/'clients_clean.csv',encoding='utf-8-sig',dtype={'client_id':'string','branch_code':'string'})
transactions=pd.read_csv(folder/'transactions_clean.csv',encoding='utf-8-sig',dtype={'client_id':'string','txn_id':'string','mcc_code':'string'})
campaign=pd.read_csv('campaign_list.csv',encoding='utf-8-sig',dtype={'client_id':'string','group':'string'})
cleaning_log=pd.read_csv('cleaning_log.csv',encoding='utf-8-sig')
for col in ['birth_date','registration_date','card_open_date','app_last_login']:
    clean[col]=pd.to_datetime(clean[col])
transactions['txn_datetime']=pd.to_datetime(transactions.txn_datetime)
for col in ['activity_comparable','activity_declined','contact_allowed6','persistent_decline6','card_loss_not_compensated6']:
    clean[col]=flag(clean[col])
indexed=clean.set_index('client_id')
fixed=clean.loc[clean.activity_comparable].copy()
positive=clean.loc[clean.activity_comparable & clean.payments_base.gt(0)].copy()
months=pd.period_range('2024-01','2025-06',freq='M')

# Нужны только успешные оплаты; остальные статусы остаются в transactions.
payments=transactions.loc[transactions.status.eq('успешно') & transactions.amount.gt(0)
    & np.isfinite(transactions.amount)].copy()
payments['month']=payments.txn_datetime.dt.to_period('M')
payments['period']=np.where(payments.txn_datetime.lt('2025-01-01'),'2024','2025')
monthly=(payments.loc[payments.client_id.isin(indexed.index)]
    .groupby(['client_id','month'],observed=True).size().unstack(fill_value=0)
    .reindex(index=indexed.index,columns=months,fill_value=0).fillna(0).astype(int))
base_mean=fixed.activity_base.mean()
current_mean=fixed.activity_current.mean()
annual_change=100*(current_mean/base_mean-1) if base_mean>0 else np.nan
h1_base=monthly.loc[fixed.client_id,'2024-01':'2024-06'].to_numpy().sum()/6/len(fixed)
h1_current=monthly.loc[fixed.client_id,'2025-01':'2025-06'].to_numpy().sum()/6/len(fixed)
h1_change=100*(h1_current/h1_base-1) if h1_base>0 else np.nan
n_declined=int(positive.activity_declined.sum())
share_declined=100*n_declined/len(positive)
unlinked=transactions.loc[~transactions.client_id.isin(indexed.index)]

# Сначала сравниваем всех массовых клиентов, не только выбранных для обращения.
mass=positive.loc[positive.segment.astype('string').eq('Массовый').fillna(False)].copy()
mass['app_group']=mass.has_mobile_app.astype('string').fillna('Нет данных')
app_summary=mass.groupby('app_group').agg(n=('client_id','size'),
    base=('activity_base','mean'),current=('activity_current','mean'),q2=('q2_frequency6','mean'),
    declined=('activity_declined','mean'))
app_summary['declined_pct']=100*app_summary.declined

# Каналы сравниваем на одном и том же составе клиентов с приложением.
app_ids=positive.loc[positive.has_mobile_app.astype('string').eq('Да').fillna(False),'client_id']
part=payments.loc[payments.client_id.isin(app_ids)]
counts=part.groupby(['channel','period'],observed=True).size().unstack(fill_value=0).reindex(columns=['2024','2025'],fill_value=0)
channels=pd.DataFrame({'base':counts['2024']/12/(len(app_ids) or np.nan),
                       'current':counts['2025']/6/(len(app_ids) or np.nan)})
channels['change']=channels.current-channels.base
channels=channels.sort_values('change')
loss=-channels.change.clip(upper=0).sum()
growth=channels.change.clip(lower=0).sum()
coverage=100*growth/loss if loss>0 else np.nan

# Смотрим повседневное снижение до фильтра по изменению суммы.
everyday_basis=positive.loc[positive.everyday_change6.lt(0)
    & positive.everyday_loss_share6.ge(.5) & positive.rub_month_base6.gt(0)].copy()
everyday=['Продукты','Транспорт','Аптеки','Связь и Интернет']
part=payments.loc[payments.client_id.isin(everyday_basis.client_id) & payments.category.isin(everyday)]
counts=part.groupby('period').size().reindex(['2024','2025'],fill_value=0)
everyday_frequency=np.array([counts['2024']/12,counts['2025']/6])/(len(everyday_basis) or np.nan)
everyday_money=everyday_basis[['rub_month_base6','rub_month_current6']].mean().to_numpy()
money_down=int(everyday_basis.rub_month_change6.lt(0).sum())
money_down_pct=100*money_down/len(everyday_basis) if len(everyday_basis) else np.nan

group_sizes=campaign.groupby('group').size()
selected=campaign.merge(clean,on='client_id',how='left',validate='one_to_one')
campaign_valid=bool(3<=campaign.group.nunique()<=5 and len(campaign)<=2000
    and campaign.client_id.is_unique and campaign.client_id.isin(indexed.index).all()
    and group_sizes.ge(150).all() and selected.contact_allowed6.fillna(False).all())
metric_match=bool(np.allclose(monthly.loc[clean.client_id,'2024-01':'2024-12'].sum(axis=1).to_numpy(),clean.payments_base)
    and np.allclose(monthly.loc[clean.client_id,'2025-01':'2025-06'].sum(axis=1).to_numpy(),clean.payments_current))

## 1. Главное

In [ ]:
app_text='; '.join(f'{name}: {fmt(row.declined_pct)}% из {int(row.n)}' for name,row in app_summary.iterrows())
display(Markdown(f"""
1. У {len(fixed):,} клиентов за январь 2024 — июнь 2025 учтено {int(monthly.loc[fixed.client_id].to_numpy().sum()):,} успешных оплат. Среднее число оплат в месяц изменилось с {fmt(base_mean,2)} до {fmt(current_mean,2)} ({fmt(annual_change)}%). При одинаковых первых полугодиях изменение — {fmt(h1_change)}%. Реже платить стали {n_declined:,} из {len(positive):,} клиентов с оплатами в 2024 году ({fmt(share_declined)}%).
2. В массовом сегменте доля ставших платить реже различается по наличию приложения: {app_text}. 
3. У {len(app_ids):,} клиентов с приложением больше всего сократилось число оплат через «{channels.change.idxmin()}» — в среднем на {fmt(-channels.change.min(),3)} оплаты на клиента в месяц. Суммарное сокращение в каналах, где оплат стало меньше, составило {fmt(loss,4)} оплаты на клиента в месяц, а суммарный рост в остальных — {fmt(growth,4)}. Этот рост компенсировал {fmt(coverage,2)}% сокращения.
4. Среди {len(everyday_basis)} клиентов, у которых повседневные категории дают не менее половины всех сокращений частоты, месячная сумма RUB-оплат также упала у {money_down} ({fmt(money_down_pct)}%). Значит, снижение частоты не всегда означает снижение суммы.
5. Для обращения выбрано {len(campaign):,} клиентов, разделённых на {campaign.group.nunique()} группы. Их состав и правила приведены в разделе 5.
"""))
if not campaign_valid or not metric_match:
    display(Markdown('Есть расхождения в данных или списке клиентов. Перед использованием отчёта повторите анализ и экспорт.'))
    
    display(channels[['base', 'current', 'change']].round(4))


## 2. Контекст
Банк хочет вернуть активность клиентов. Нужно выбрать до 2000 человек и разделить их на 3–5 понятных групп не меньше 150 клиентов каждая. Обращаться можно только к согласившимся клиентам. Задача анализа — показать, кто стал платить реже и какие оплаты сократились. 

## 3. Данные и ограничения
Использованы анкеты действующих клиентов на 30.06.2025 и операции за январь 2024 — июнь 2025. Отчёт читает очищенные данные и результаты первого notebook.

Активность — среднее число успешных оплат на клиента в месяц. Количество за 2024 год делим на 12, за первое полугодие 2025 — на 6. Учитываем положительные конечные суммы. Отклонённые операции и возвраты в метрику не входят.

Для основного сравнения берём зарегистрированных не позднее 01.01.2024. Более поздних клиентов сохраняем, но исследуем отдельно: у них нет данных за полный 2024 год. Месяцы без оплат считаем нулевыми.

Метрика не показывает сумму покупок или причины снижения. Число оплат учитывает все каналы и валюты; суммы отдельно сравниваем в рублях без вычитания возвратов. Покупки вне банка и полная история ушедших клиентов неизвестны.

In [ ]:
display(Markdown(f"""
- После очистки осталось {len(clean):,} анкет и {len(transactions):,} операций периода всех статусов.
- Для {len(unlinked):,} операций ({fmt(100*len(unlinked)/len(transactions))}%) не найдена анкета клиента.
- У {int((~clean.activity_comparable).sum())} клиентов нет полной истории для сравнения; ещё у {int((clean.activity_comparable & clean.payments_base.eq(0)).sum())} сопоставимых клиентов не было оплат в 2024 году.
- Доход не указан у {int(clean.monthly_income.isna().sum()):,} из {len(clean):,} клиентов, оценка — у {int(clean.satisfaction_score.isna().sum()):,} из {len(clean):,}.
"""))

## 4. Находки

### 4.1. В массовом сегменте сравниваем клиентов с приложением и без
Учитываем всех массовых клиентов с полной историей и оплатами в 2024 году. Слева — среднее число оплат, справа — доля ставших платить реже. Это сравнение клиентов, а не оценка эффекта установки приложения.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(9,2.8))
view=app_summary[['base','current','q2']].rename(columns={'base':'2024','current':'I полугодие 2025','q2':'апрель-июнь 2025'})
view.index=[f'{name} (n={int(app_summary.loc[name,"n"])})' for name in view.index]
view.plot.bar(ax=axes[0],rot=0,color=['#3c78a8','#ba4d51','#65947b'])
axes[0].set(title='Среднее число оплат',xlabel='Есть приложение',ylabel='Оплат / клиента / месяц')
axes[0].legend(fontsize=7)
bars=axes[1].bar(view.index,app_summary.declined_pct,color='#547ca5')
axes[1].bar_label(bars,fmt='%.1f%%',padding=3)
axes[1].set(title='Кто стал платить реже',xlabel='Есть приложение',ylabel='Клиентов группы, %',ylim=(0,110))
fig.tight_layout();plt.show()
# Подпись использует текущий результат, а не заранее заданный вывод.
known=app_summary.reindex(['Нет','Да'])
if known.n.notna().all():
    gap=known.loc['Нет','declined_pct']-known.loc['Да','declined_pct']
    display(Markdown(f'Без приложения реже платят {fmt(known.loc["Нет","declined_pct"])}%, с приложением — {fmt(known.loc["Да","declined_pct"])}%. Отдельно показан апрель-июнь 2025: он позволяет увидеть, сохраняется ли снижение к концу периода. '
        'Другие различия между клиентами в этом сравнении не устранены.'))

### 4.2. Рост одних способов оплаты может компенсировать сокращение других
Сравниваем одних и тех же клиентов с приложением, полной историей и оплатами за 2024. Для каждого канала считаем изменение среднего числа оплат в месяц. 

In [ ]:
fig,ax=plt.subplots(figsize=(8,2.6))
bars=ax.barh(channels.index.astype(str),channels.change,color=np.where(channels.change.lt(0),'#ba4d51','#3c78a8'))
ax.bar_label(bars,fmt='%+.2f',padding=3)
ax.axvline(0,color='gray',linewidth=.8)
ax.set(title=f'Клиенты с приложением: n={len(app_ids)}',xlabel='Изменение числа оплат на клиента в месяц')
ax.margins(x=.3);fig.tight_layout();plt.show()
display(Markdown(f'Рост каналов покрывает {fmt(coverage)}% потери частоты в сокращающихся каналах. '
    f'Общее изменение на этой выборке — {fmt(channels.change.sum(),3)} оплаты на клиента в месяц. '
    'Это общий итог: у отдельных клиентов соотношение может быть другим. '
    'Одновременный рост одного канала и падение другого не доказывают перенос тех же покупок.'))

### 4.3. Меньше повседневных оплат — не всегда меньше потраченных рублей
Рассматриваем клиентов, у которых стало меньше оплат продуктов, транспорта, аптек, связи и интернета. На эти категории приходится не менее половины общего сокращения числа оплат. В 2024 году у этих клиентов были покупки в рублях. Затем проверяем, стали ли они тратить меньше. Для этого оставляем в сравнении и тех, у кого сумма расходов не снизилась.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(9,2.7))
# Состав клиентов на двух графиках одинаковый, единицы разные.
for ax,values,title,ylabel in [
    (axes[0],everyday_frequency,'Повседневные оплаты','Оплат / клиента / месяц'),
    (axes[1],everyday_money,'Все успешные RUB-оплаты','Рублей / клиента / месяц'),
]:
    bars=ax.bar(['2024','I пол. 2025'],values,color=['#3c78a8','#ba4d51'])
    ax.bar_label(bars,fmt='%.1f',padding=3)
    ax.set(title=title,ylabel=ylabel);ax.margins(y=.2)
fig.suptitle(f'Одна и та же группа: n={len(everyday_basis)}',fontsize=11)
fig.tight_layout();plt.show()
display(Markdown(f'Месячная сумма RUB-оплат уменьшилась у {money_down} из '
    f'{len(everyday_basis)} клиентов ({fmt(money_down_pct)}%). '
    'Поэтому изменение количества и суммы оплат рассматриваем отдельно. '
    'Суммы указаны в текущих рублях, без вычитания возвратов; '
    'по ним нельзя установить изменение дохода или реальных потребностей.'))

## 5. Группы для кампании
Для всех групп нужны полная история, оплаты в начале периода, снижение средней месячной частоты за полугодие и во II квартале относительно 2024 года, явное согласие.

- G1 — массовый сегмент без приложения. Обсудить удобство привычных оплат без обязательной установки приложения.
- G2 — есть приложение; карта даёт не менее половины всех сокращений частоты, её потерю не покрывает рост других каналов. Выяснить, что мешает пользоваться способами оплаты.
- G3 — клиенты, у которых стало меньше оплат продуктов, транспорта, аптек, связи и интернета. На эти категории приходится не менее половины общего сокращения числа оплат. При этом средняя сумма всех оплат в рублях за месяц тоже снизилась по сравнению с 2024 годом, когда у клиента были рублёвые оплаты. 

До назначения клиент может подходить под несколько правил. В итоговом списке он один раз: приоритет G1 → G2 → G3. Размеры пересекающихся кандидатов нельзя складывать. Выяснить, какое предложение для повседневных покупок будет полезно клиенту.

In [ ]:
# Только описываем правила и сравниваем с готовым списком; новых назначений нет.
common=positive.activity_declined & positive.persistent_decline6
app=positive.has_mobile_app.astype('string')
segment=positive.segment.astype('string')
rule_masks=pd.DataFrame({
    'G1':common & segment.eq('Массовый').fillna(False) & app.eq('Нет').fillna(False),
    'G2':common & app.eq('Да').fillna(False) & positive.card_loss_not_compensated6 & positive.card_loss_share6.ge(.5),
    'G3':common & positive.everyday_change6.lt(0) & positive.everyday_loss_share6.ge(.5)
        & positive.rub_month_base6.gt(0) & positive.rub_month_change6.lt(0),
})
rule_sizes=pd.DataFrame({'До согласия':rule_masks.sum(),
    'С согласием':rule_masks.mul(positive.contact_allowed6,axis=0).sum()})
rule_sizes['Выбрано']=group_sizes.reindex(rule_sizes.index,fill_value=0)
show_table(rule_sizes.rename_axis('Правило'))
groups=selected.groupby('group').agg(n=('client_id','size'),base=('activity_base','mean'),
    current=('activity_current','mean'),q2=('q2_frequency6','mean'),
    rub_base=('rub_month_base6','mean'),rub_current=('rub_month_current6','mean'))
show_table(groups.rename(columns={'n':'Клиентов','base':'Оплат/мес. 2024','current':'Оплат/мес. 2025',
    'q2':'Оплат/мес. II кв.','rub_base':'RUB/мес. 2024','rub_current':'RUB/мес. 2025'}))

display(Markdown('Во второй таблице показаны только выбранные клиенты. В каждой группе сначала выбираем тех, у кого сильнее сократилось среднее число оплат в месяц. Если сокращение одинаковое — тех, у кого число оплат за апрель–июнь сильнее отстаёт от уровня 2024 года. При полном совпадении сортируем по client_id. Сначала выделяем места для минимального размера каждой группы, затем распределяем оставшиеся.'))

## 6. Кого не включаем и почему
Не включаем клиентов без полной истории, без оплат в 2024 году, без снижения или явного согласия, а также восстановивших среднее число оплат в месяц за 2024 год во II квартале. Затем применяются правила и бюджет. Каждой анкете назначена одна основная причина. Из очищенных данных клиенты не удаляются.

In [ ]:
excluded=clean.decision.astype('string').fillna('Решение не записано').value_counts().to_frame('Клиентов')
excluded['Доля всех анкет, %']=100*excluded['Клиентов']/len(clean)
show_table(excluded,1)


## 7. Что проверили и что не подтвердилось

In [ ]:
# Отказы считаем среди попыток; возвраты в знаменатель не входят.
attempts=transactions.loc[transactions.client_id.isin(positive.client_id)
    & transactions.channel.eq('Карта') & transactions.status.isin(['успешно','отклонено'])].copy()
attempts['year']=attempts.txn_datetime.dt.year
refusal=attempts.assign(refused=attempts.status.eq('отклонено')).groupby('year').refused.mean()*100
# У этой когорты первое полугодие 2024 — до 65-летия, первое 2025 — после.
birthday65=clean.birth_date.map(lambda d:d+pd.DateOffset(years=65) if pd.notna(d) else pd.NaT)
senior_ids=clean.loc[clean.registration_date.le('2024-01-01')
    & birthday65.ge('2024-07-01') & birthday65.lt('2025-01-01'),'client_id']
new_ids=clean.loc[clean.registration_date.ge('2024-07-01') & clean.registration_date.lt('2025-07-01'),'client_id']
senior_delta=int(monthly.loc[senior_ids,'2025-01':'2025-06'].to_numpy().sum()
    -monthly.loc[senior_ids,'2024-01':'2024-06'].to_numpy().sum())
new_count=int(monthly.loc[new_ids,'2025-01':'2025-06'].to_numpy().sum())
# Сравниваем город внутри одинаковых сегмента и лояльности.
geo=positive.loc[positive.city.notna()].copy()
geo['region']=np.where(geo.city.eq('Екатеринбург'),'Екатеринбург','Остальные')
for col in ['segment','loyalty_tier']:geo[col]=geo[col].astype('string').fillna('Нет данных')
geo['b']=monthly.loc[geo.client_id,'2024-01':'2024-06'].sum(axis=1).to_numpy()/6
geo['c']=monthly.loc[geo.client_id,'2025-01':'2025-06'].sum(axis=1).to_numpy()/6
strata=geo.groupby(['segment','loyalty_tier','region'],observed=True).agg(n=('b','size'),b=('b','mean'),c=('c','mean'))
strata['change']=100*(strata.c/strata.b.replace(0,np.nan)-1)
values=strata.change.unstack().reindex(columns=['Екатеринбург','Остальные'])
sizes=strata.n.unstack().reindex_like(values)
difference=(values['Екатеринбург']-values['Остальные']).loc[sizes.ge(30).all(axis=1)&values.notna().all(axis=1)]
score=pd.to_numeric(positive.satisfaction_score,errors='coerce')
branches=positive.assign(rated=score.notna()).groupby('branch_code').agg(n=('rated','size'),rated=('rated','sum'))
enough=int((branches.n.ge(30)&branches.rated.ge(20)).sum())
display(Markdown(f"""
- Разные окна: при одинаковых первых полугодиях изменение {fmt(h1_change)}%. Отрицательный результат не объясняется одним выбором годовой базы.
- Отказы при оплате картой: в 2024 году отклонено {fmt(refusal.get(2024,np.nan),3)}% попыток оплаты, в январе–июне 2025 года — {fmt(refusal.get(2025,np.nan),3)}%. Сравниваем клиентов с полной историей, у которых были оплаты в 2024 году. Общий показатель может скрывать проблемы у отдельных клиентов.
- Старение и приток: у {len(senior_ids)} достигших 65 во II пол. 2024 изменение {senior_delta:+d} оплат; новые всех возрастов последних двух полугодий добавили {new_count}. Частичный итог {senior_delta+new_count:+d}; это не изменение всего банка и не доказательство влияния возраста.
- Екатеринбург: для сравнения с другими городами подошли {len(difference)} групп клиентов с одинаковыми сегментом и уровнем лояльности, но ним нельзя сделать вывод обо всех клиентах города.
- Обслуживание: у {fmt(100*score.isna().mean())}% клиентов нет оценки обслуживания, поэтому не хватает данных для вывода стали ли реже платить из-за обслуживания.
"""))

## 8. Приложение

### 8.1. Сводка журнала изменений
Полный журнал — `cleaning_log.csv`. Ниже число решений по темам, а не число удалённых клиентов. Одни и те же строки могут участвовать в нескольких решениях.

In [ ]:
patterns=[
    ('Типы и даты',r'Заданы типы|Преобразованы в datetime'),
    ('Названия категорий',r'Нормализация'),
    ('Повторы и версии',r'Удалены полные|Удалена полуночная|Объединены версии'),
    ('Противоречивые поля',r'Заменено пропуском'),
    ('Значения оставлены',r'Оставлено'),
    ('Период и исключения по датам',r'Создана отдельная выборка периода|Удалено из рабочей'),
    ('Диагностики и прочие решения',r'Создан|Выбраны'),
    ('Отбор кампании',r'Применены правила'),
]
used=pd.Series(False,index=cleaning_log.index)
rows=[]
for label,pattern in patterns:
    mask=cleaning_log.action.astype('string').str.contains(pattern,case=False,regex=True,na=False)&~used
    part=cleaning_log.loc[mask];used|=mask
    rows.append({'Решения':label,'Записей':len(part),
        'Примеры полей':'; '.join(part.field.astype(str).head(3))[:100] or '—'})
rows[6]['Записей']+=int((~used).sum())
show_table(pd.DataFrame(rows).set_index('Решения'),0)

### 8.2. Методика
- Период: `[01.01.2024, 01.07.2025)`. Успешная оплата: статус «успешно», сумма положительна и конечна. Делители — 12/6 месяцев, для II квартала — 3. Основная выборка: известная регистрация ≤ 01.01.2024; индивидуальные проценты — при положительной базе.
- Снижение — текущая частота строго меньше частоты за 2024. В кампании частота II квартала тоже ниже 2024 года, есть явное согласие «да».
- 50% означает «не менее половины всех сокращений частоты», а не падение активности на 50% и не долю исходных покупок. Потеря канала — модуль отрицательного изменения. Для G2 потеря карты / сумма потерь каналов ≥ 0,5; сумма положительных изменений других каналов меньше потери карты.
- Для G3 потери продуктов, транспорта, аптек, связи и интернета / сумма потерь категорий ≥ 0,5. Суммарная частота этих категорий и общая месячная RUB-сумма падают, RUB-база положительная. Возвраты из сумм не вычитаются.
- Пропуски дохода и оценок не заполнены нулями. Полнота связана с известными признаками. Подозрительные доходы сохранены; для описания в анализе используются медиана и квартили.
- Возрастной баланс частичный. Для географии n ≥ 30 с каждой стороны — ограничение малых ячеек.

### 8.3. Использование ИИ
ИИ помогал с формулировками, написанием кода, структурированием, что лучше в какой главе описать.
Советовалась при выборе типа графика для визуализации данных.
Обсуждала с ИИ некоторые гипотезы.
Активно его использовала для видения журнала.